<h1 style = "color : #0EE071; text-align : center;"><em>Where should I live?</em> - Building an Interactive Map Notebook</h1>
<p style = "font-size : 16px; text-align: center;">The goal of this section is to create an interactive map of Europe where users can explore cities and view relevant information</p>
<br>
<p style = "font-size : 12px; text-align: center;"><b>NOVA IMS</b></p>
<p style = "font-size : 10px; text-align: center;">Programming for Data Science</p>
<p style = "font-size : 10px; text-align: center;">Diogo Gonçalves, João Marques, Juan Mendes & Gustavo Franco</p>
<br>

<h2  style = "color : #0EE071;"> Imports</h2>

In [37]:
import pandas as pd
import plotly.graph_objects as go
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import warnings
warnings.filterwarnings('ignore')

In [38]:
#!pip install requests

In [39]:
#!pip install beautifulsoup4

In [40]:
#!pip install selenium

<hr style = "border: 3px solid #0EE071;">
<h2 style = "color : #0EE071;">Dataset Importing & Preparation</h2>
<p style = "font-size : 15px;">Reading of dataset from <code>city_data_clean.csv</code> file that we have prepared previously.</p>

In [41]:
city_data = pd.read_csv("city_data_clean.csv")
city_data.head()

,Population Density,Population,Working Age Population,Youth Dependency Ratio,Unemployment Rate,GDP per Capita,Days of very strong heat stress,Average Monthly Salary,Average Rent Price,Average Cost of Living,...,Scots,Serbian,Slovak,Slovene,Spanish,Swedish,Turkish,Unknown,Urdu,Valencian
0,310.0,2983513,2018818,20.1,10.2,55770.0,3,2500,1050,2061,...,0,1,0,0,0,0,1,0,0,0
1,243.0,375489,250472,20.3,3.0,66689.0,0,3200,1100,2186,...,0,0,0,0,0,0,0,0,0,0
2,681.0,3284548,2137425,27.5,10.7,62500.0,3,3350,1200,1900,...,0,0,0,0,0,0,0,0,0,0
3,928.0,1139663,723396,27.7,6.2,57595.0,3,2609,900,1953,...,0,0,0,0,0,0,0,0,0,0
4,552.0,645813,417832,24.8,5.3,53311.0,2,2400,827,1200,...,0,0,0,0,0,0,0,0,0,0


In [42]:
imdf = city_data[['City', 'Country', 'Population', 'Average Monthly Salary', 'Average Cost of Living']]
imdf.head()

,City,Country,Population,Average Monthly Salary,Average Cost of Living
0,Vienna,Austria,2983513,2500,2061
1,Salzburg,Austria,375489,3200,2186
2,Brussels,Belgium,3284548,3350,1900
3,Antwerp,Belgium,1139663,2609,1953
4,Gent,Belgium,645813,2400,1200


<hr style = "border: 3px solid #0EE071;">
<h2  style = "color : #0EE071;">Web Scraping</h2>
<p style="font-size: 15px;">
  <span style="font-size: 20px;">Short summary:</span>
  <br><br>
  - <code>requests</code> : download HTML only<br>
  - <code>BeautifulSoup</code> : extract info from that HTML<br>
  - <code>Selenium</code> : control a browser and interact with the page
</p>

In [43]:
imdf["Latitude"] = None
imdf["Longitude"] = None

In [44]:
def get_coordinates(browser, city, country):
        # Find search icon on browser and click it
        search_icon_input = browser.find_element(By.CLASS_NAME, "mw-ui-icon-search")
        search_icon_input.click()
        time.sleep(1)
        
        # Find search bar on browser, insert city and click enter
        search_bar_input = browser.find_element(By.CLASS_NAME, "cdx-text-input__input")
        search_bar_input.send_keys(f"{city}, {country} city")
        search_bar_input.send_keys(Keys.RETURN)
        time.sleep(2)
        
        html = browser.page_source
        readable_html = BeautifulSoup(html, "html.parser")
    
        latitude = readable_html.find("span", {"class": "latitude"})
        longitude = readable_html.find("span", {"class": "longitude"})
        
        links = browser.find_elements(By.CSS_SELECTOR, ".mw-search-result-heading a")
    
        max_tries = 5
        
        for link in links[:max_tries]:
            link.click()
            time.sleep(2)
    
            soup = BeautifulSoup(browser.page_source, "html.parser")
            latitude = soup.find("span", class_="latitude")
            longitude = soup.find("span", class_="longitude")
    
            if latitude and longitude:
                return latitude.text, longitude.text
    
            browser.back()
            time.sleep(2)

In [45]:
def get_coordenates_insert(data):
    # Open Google Chrome and search Wikipedia
    browser = webdriver.Chrome()
    browser.get('https://en.wikipedia.org/wiki/Main_Page')
    time.sleep(1)

    # Loop to insert the coordenates in the dataset
    for idx in data.index:
        city = data.loc[idx, "City"]
        country = data.loc[idx, "Country"]

        result = get_coordinates(browser ,city, country)
        if result:
            latitude, longitude = result
        else:
            latitude, longitude = None, None

        data.loc[idx, "Latitude"] = latitude
        data.loc[idx, "Longitude"] = longitude

    browser.quit()

In [46]:
get_coordenates_insert(imdf)

In [47]:
imdf.head()

,City,Country,Population,Average Monthly Salary,Average Cost of Living,Latitude,Longitude
0,Vienna,Austria,2983513,2500,2061,48°12′30″N,16°22′21″E
1,Salzburg,Austria,375489,3200,2186,47°48′00″N,13°02′42″E
2,Brussels,Belgium,3284548,3350,1900,50°50′48″N,04°21′09″E
3,Antwerp,Belgium,1139663,2609,1953,51°13′04″N,04°24′01″E
4,Gent,Belgium,645813,2400,1200,51°03′13″N,03°43′31″E


In [48]:
imdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   City                    84 non-null     object
 1   Country                 84 non-null     object
 2   Population              84 non-null     int64 
 3   Average Monthly Salary  84 non-null     int64 
 4   Average Cost of Living  84 non-null     int64 
 5   Latitude                84 non-null     object
 6   Longitude               84 non-null     object
dtypes: int64(3), object(4)
memory usage: 4.7+ KB


<hr style = "border: 3px solid #0EE071;">
<h2 style = "color : #0EE071;">Interactive Map</h2>
<p style = "font-size : 16px;">It displays our data interactively on the map</p>
<br>

In [49]:
# This function turns the coordinates written like (51° 3′ 13″ N, 3° 43′ 31″ E) into 
def interactive_coordinates(str_coordinates):
    if str_coordinates is None:
        return None

    try:
        direction = str_coordinates[-1]

        core = str_coordinates[:-1]

        degrees_part, rest = core.split("°")
        degrees = float(degrees_part)

        minutes_part, rest = rest.split("′")
        minutes = float(minutes_part)

        seconds_part = rest.replace("″", "")
        seconds = float(seconds_part)

        dd = degrees + minutes/60 + seconds/3600

        if direction in ("S", "W"):
            dd = -dd

        return dd

    except:
        return None

In [50]:
imdf["Interactive Latitude"] = imdf["Latitude"].apply(interactive_coordinates)
imdf["Interactive Longitude"] = imdf["Longitude"].apply(interactive_coordinates)

In [51]:
fig = go.Figure()

fig.add_trace(go.Scattergeo(
    lon=imdf["Interactive Longitude"],
    lat=imdf["Interactive Latitude"],
    text=imdf["City"], 
    mode="markers+text",
    textposition="bottom center",
    marker=dict(size=6, color="blue"),
    customdata = imdf[["Country", "Population", "Average Monthly Salary", "Average Cost of Living"]].values,
    hovertemplate=(
        "<b>%{text}</b><br>" +
        "Country = %{customdata[0]}<br>" +
        "Population = %{customdata[1]}<br>" +
        "Average Monthly Salary = %{customdata[2]}<br>" +
        "Average Cost of Living = %{customdata[3]}<extra></extra>"),))


fig.update_layout(
        geo=dict(
        scope="world",
        projection_type="mercator",
        showcountries=True,
        countrycolor="lightgray",
        lataxis_range=[30, 72],
        lonaxis_range=[-15, 45], 
    ),
    height=600
)

fig.show()